# Reproducing Silkuniene et al. (2025) from dandiset 001276

**Paper.** Silkuniene G, Silkunas M, Pakhomova ON, Semenov I, Pakhomov AG (2025). MHz compression of
pulse packets facilitates remote focusing of electroporation. *Bioelectrochemistry* 166:109016.
[doi:10.1016/j.bioelechem.2025.109016](https://doi.org/10.1016/j.bioelechem.2025.109016). The claims
file was written from the accepted author manuscript in PubMed Central (PMC12331426), with its
Methods and figure legends; the paper has no supplement and links no code.

**Dandiset.** [DANDI:001276](https://dandiarchive.org/dandiset/001276/0.250701.1609), version
0.250701.1609. The paper cites no version of the dandiset (its data statement says "Data will be made
available on request"). This is the only published version, and the most recent; the draft has the
same 108 assets and size, so there is nothing to choose between.

**What the dandiset holds.** One stitched whole-well image per file: YO-PRO-1 (YP) after exposure, or
Hoechst before it, for 54 wells of CT26 and CHO monolayers. The wells are the burst-number series of
Fig. 7 (0.2 MHz, 600 ns pulses, 1, 2, 4 or 6 bursts (x4)), and in addition CANCAN and control wells at
a packet rate of 1 Hz, 2 bursts (x4). There are no 1 kHz or 10 kHz wells, no raised electrodes, no
propidium images, no 300 or 900 ns pulses and no thermometry, so the claims on those are blocked. Three
other dandisets of the same lab cite this paper and, by their titles and descriptions, hold those series:
DANDI:001047 (1 Hz against 0.2 MHz, with propidium), DANDI:001277 (300, 600 and 900 ns pulses) and
DANDI:001284 (electrodes raised 0, 1 or 2 mm, with propidium). The claims file names no data outside this
dandiset, so they are not used here; nothing in 001276 points to them.

| Claim | Tested here |
|---|---|
| fig7be-center-linear-with-bursts | tested |
| fig7b-ct26-saturation | tested |
| fig7cf-fwhm-with-bursts | tested |
| fig7-sharpest-peak-one-burst | tested |
| fig7-offtarget-40-50-fold | tested |
| fig7-cho-replicates-ct26 | tested |
| fig4b-40-fold-at-02mhz | tested |
| fig3a-cancan-center-enhanced | tested on the rates present (1 Hz, 0.2 MHz, control) |
| fig3bc-lorentzian-profile | tested on the rates present (1 Hz, 0.2 MHz) |
| fig3bc-offtarget-deviation-falls-with-rate | tested (1 Hz against 0.2 MHz) |
| fig4b-center-no-rate-effect | tested on the rates present (0.2 MHz against 1 Hz) |
| fig4b-near-electrode-rate-decline | tested on the rates present (1 Hz, 0.2 MHz) |
| fig4c-fwhm-decreases-with-rate | tested on the rates present (1 Hz, 0.2 MHz) |
| fig4c-log-dependence | blocked: no 1 kHz or 10 kHz wells (a rate series is in DANDI:001047) |
| fig5-raised-electrodes-weaker-center | blocked: no raised-electrode wells (in DANDI:001284) |
| fig5b-pr-gaussian-yp-lorentzian | blocked: no propidium images (in DANDI:001284) |
| fig5-yp-spot-slightly-larger-than-pr | blocked: no propidium images (in DANDI:001284) |
| fig5c-irreversible-center | blocked: no post-exposure Hoechst or propidium images (in DANDI:001284); the YP part is shown |
| fig6b-900ns-peak-not-significant | blocked: no 300 or 900 ns wells (in DANDI:001277) |
| fig6c-normalized-decline-similar | blocked: no 300 or 900 ns wells (in DANDI:001277) |
| fig6d-fwhm-sharper-longer-pulses | blocked: no 300 or 900 ns wells (in DANDI:001277) |
| fig2-heating-below-44c | blocked: no thermochromic film images |
| fig1e-field-simulation | blocked: the field model is not public |

No random numbers are used anywhere in this notebook.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats

import cancan

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = cancan.list_assets()
meta = cancan.load_metadata(assets)
meta["date"] = meta.subject.str.split("_").str[1]
meta["plate"] = meta.subject.str.split("_").str[0]
print(f"dandiset {cancan.DANDISET_ID} version {cancan.DANDISET_VERSION}: {len(assets)} assets, "
      f"{meta.size_mb.sum() / 1e3:.1f} GB; metadata read in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files by channel and phase:")
print(meta.groupby(["channel", "phase"]).size().to_string())
print("\nImage shapes, dtypes and chunking:")
print(meta.groupby(["shape", "dtype", "chunks"]).size().to_string())
wells = meta.groupby("subject").agg(
    cell_line=("cell_line", "first"), species=("species", "first"), protocol=("protocol", "first"),
    rate=("packet_rate", "first"), bursts=("bursts", "first"), code=("protocol_code", "first"),
    date=("date", "first"), plate=("plate", "first"),
    files=("path", "size"), channels=("channel", lambda c: "+".join(sorted(c))),
    consistent=("bursts", "nunique"))
assert (wells.consistent == 1).all()
print(f"\n{len(wells)} wells (subjects)")
print("\nWells by cell line, protocol, packet rate and number of bursts (x4):")
print(wells.groupby(["cell_line", "protocol", "rate", "bursts"]).size().to_string())
print("\nWells whose files are not one DAPI (pre) and one FITC (post):")
print(wells[wells.channels != "DAPI+FITC"][["cell_line", "protocol", "rate", "bursts", "channels"]].to_string())
print("\nCell line against species:")
print(wells.groupby(["cell_line", "species"]).size().to_string())
print("\nProtocol code against protocol and rate:", wells.groupby(["code", "protocol", "rate"]).size().to_dict())
print("\nWells per experiment date (from the subject id) and cell line:")
print(pd.crosstab(wells.date, wells.cell_line).to_string())
print("\nsession_start_time values (dates):", sorted(meta.session_start.str[:10].unique()))
print("Inter-electrode distance stated in every description:",
      sorted(set(meta.description.str.extract(r"inter-electrode distance of ([\d.]+ mm)")[0])))

The dandiset holds 54 wells (its "subjects"), each a well of a 6-well plate. Cell line and species agree
(CT26 with *Mus musculus*, CHO with *Cricetulus griseus*). The protocol, packet rate and number of bursts
are only in the free-text description, where code 011 is 0.2 MHz and code 013 is 1 Hz. Compared with the
paper's Fig. 7 (CT26 n = 3 to 5, CHO n = 4), the dandiset has 3, 6, 3 and 3 CT26 wells and 5, 8, 7 and 8
CHO wells for 1, 2, 4 and 6 bursts (x4), more CHO wells than the paper reports, with no record of which
wells the paper used. It also has CANCAN and control wells at 1 Hz, 2 bursts (x4): CT26 (2 and 4 wells,
the paper has n = 3 and n = 5 controls for Fig. 3 and 4) and CHO (3 and 2 wells, not in the paper). Every
`session_start_time` is 2024-12-12, the conversion date; the experiment dates are only in the subject ids
(June to September 2024). One well has no Hoechst file and one has two (below). The stated
"inter-electrode distance of 10.0 mm" fits the paper's geometry only as the edge-to-edge distance along a
side (12.8 mm between centers less 2.8 mm); the electrode fits below give 12.8 mm between centers.

In [ ]:
odd = meta[(meta.channel == "DAPI") & meta.subject.isin(["P1_20240821_A1", "P1_20240821_A2"])]
rows = []
for _, r in odd.iterrows():
    planes = r["shape"][1] if len(r["shape"]) == 4 else 1
    for plane in range(planes):
        b = cancan.load_binned(r.url, plane=plane)
        im = b["image"]
        rows.append({"file": r.path.split("/")[-1], "shape": r["shape"], "size_mb": round(r.size_mb, 1),
                     "plane": plane, "fraction_of_area_imaged": round(float(np.mean(b["inside"] > 0)), 3),
                     "median": round(float(np.nanmedian(im)), 1), "p99": round(float(np.nanpercentile(im, 99)), 1)})
        if r.path.endswith("fkppt9_image.nwb") and plane == 0:
            frag = im
        if r.path.endswith("10gndju_image.nwb"):
            full = im
print(pd.DataFrame(rows).to_string(index=False))
m = np.isfinite(frag) & np.isfinite(full)
print(f"\nThe 12.5 MB file of P1_20240821_A2 covers {m.sum()} bins; there its plane 0 correlates with the full "
      f"Hoechst image of the same well at r = {np.corrcoef(frag[m], full[m])[0, 1]:.3f}")

Two Hoechst files have a second image plane (shape (1, 2, H, W)) that the description does not mention;
its values (median about 50) are near the camera offset, so it looks like an empty channel. Well
P1_20240821_A2 has a second Hoechst file that covers 1% of the well and matches the same region of its
full Hoechst image: a fragment of the same acquisition deposited twice. The Hoechst images are not used
by any tested claim (they were taken before exposure); they are not read further.

Next, every YP image is streamed and reduced to 10 x 10 pixel bins (16.25 um) as it loads.

In [ ]:
yp_files = meta[meta.channel == "FITC"].set_index("subject")
t0 = time.time()
images, load_info = {}, []
for subject, r in yp_files.iterrows():
    b = cancan.load_binned(r.url)
    images[subject] = b["image"]
    im = b["image"]
    load_info.append({"subject": subject, "MB": round(b["bytes"] / 1e6), "chunks": b["n_chunks"],
                      "imaged_fraction": round(float(np.mean(b["inside"] > 0)), 3),
                      "bin_min": float(np.nanmin(im)), "bin_median": float(np.nanmedian(im)),
                      "bin_max": float(np.nanmax(im))})
load_info = pd.DataFrame(load_info).set_index("subject")
print(f"{len(images)} YP images, {load_info.MB.sum() / 1e3:.1f} GB streamed in {(time.time() - t0) / 60:.1f} min")
print("\nRanges of the 10 x 10 bin means over all images (a.u., uint16 camera counts):")
print(load_info[["imaged_fraction", "bin_min", "bin_median", "bin_max"]].describe().round(1).to_string())

### Locating the electrode array

The files give neither the pixel size nor the electrode positions. The pixel size is taken as 1.625 um
(6.5 um camera pixels behind the 4x objective of Methods 2.4; an assumption, see `cancan.py`). The
electrodes are found as dark footprints ("Footprints of 4 electrodes are identified by the lack of YP
signal underneath", Fig. 3A legend): a square of the paper's size (12.8 mm between electrode centers) is
placed where its four corners are darkest against their surroundings. The fit is repeated with the side
left free, which checks the pixel size against the electrode geometry. A well is used only when the two
fits agree (centers within 0.5 mm, free side within 4% of 12.8 mm); this rule was set here, as the paper
places ROIs by eye.

In [ ]:
fits = []
squares = {}
for subject, im in images.items():
    sq = cancan.square_fit(im)
    sqf = cancan.square_fit(im, free_side=True)
    squares[subject] = sq
    fits.append({"subject": subject, "cx_mm": sq["cx"] * cancan.UM_PER_BIN / 1000,
                 "cy_mm": sq["cy"] * cancan.UM_PER_BIN / 1000, "angle_deg": sq["theta_deg"],
                 "corner_contrast_min": float(np.min(sq["corner_scores"])),
                 "free_side_mm": sqf["side_mm"],
                 "center_shift_mm": float(np.hypot(sq["cx"] - sqf["cx"], sq["cy"] - sqf["cy"]) * cancan.UM_PER_BIN / 1000)})
fits = pd.DataFrame(fits).set_index("subject").join(wells[["cell_line", "protocol", "rate", "bursts"]])
fits["used"] = (fits.center_shift_mm < 0.5) & ((fits.free_side_mm - cancan.SIDE_MM).abs() < 0.04 * cancan.SIDE_MM)
print(fits.round(3).to_string())
used = fits[fits.used]
print(f"\nWells used: {fits.used.sum()} of {len(fits)}. Left out:")
print(fits[~fits.used][["cell_line", "protocol", "rate", "bursts", "free_side_mm", "center_shift_mm"]].round(2).to_string())
print(f"\nFree side over the wells used: median {used.free_side_mm.median():.2f} mm, range "
      f"{used.free_side_mm.min():.2f} to {used.free_side_mm.max():.2f} mm (paper geometry 12.80 mm at 1.625 um per pixel)")

In [ ]:
fig, axs = plt.subplots(6, 9, figsize=(27, 18))
th = np.linspace(0, 2 * np.pi, 60)
r_fp = 1400 / cancan.UM_PER_BIN
for ax, (subject, im) in zip(axs.ravel(), images.items()):
    s = im[::4, ::4]
    ax.imshow(s, cmap="gray", vmin=np.nanpercentile(s, 1), vmax=np.nanpercentile(s, 99.7))
    sq = squares[subject]
    c = np.vstack([sq["corners"], sq["corners"][:1]]) / 4
    ax.plot(c[:, 0], c[:, 1], "r-", lw=0.6)
    for k in sq["corners"] / 4:
        ax.plot(k[0] + r_fp / 4 * np.cos(th), k[1] + r_fp / 4 * np.sin(th), "c-", lw=0.5)
    f = fits.loc[subject]
    ax.set_title(f"{subject} {f.cell_line} {f.protocol} {f.rate} {f.bursts}b" + ("" if f.used else " (left out)"), fontsize=7)
    ax.axis("off")
plt.suptitle("YP images (intensity between the 1st and 99.7th percentile of each image); red: fitted electrode square; cyan: 2.8 mm footprints")
plt.tight_layout()
plt.show()

The square of fixed size sits on the visible footprints in the wells used. In the three wells left out
(one CT26 well with 2 bursts (x4) and two CHO wells with 1 burst (x4)) the footprints are too faint, or
lie partly outside the image, for the two fits to agree. Over the wells used the free side is close to
12.8 mm, which is consistent with 1.625 um per pixel and with the paper's electrode geometry, though it
does not prove either alone.

The next check asks whether the electrode positions and the YP image share coordinates in the way the
paper describes: in CANCAN wells the YP peak should lie at the array center, and in control wells, where
the field is unipolar, YP should not peak there. The peak is the maximum of the image smoothed over 1 mm,
searched within 4 mm of the fitted center.

In [ ]:
from scipy import ndimage
rows = []
for subject in used.index:
    im = np.nan_to_num(images[subject], nan=np.nanmedian(images[subject]))
    sm = ndimage.uniform_filter(im, size=int(1000 / cancan.UM_PER_BIN))
    sq = squares[subject]
    yy, xx = np.mgrid[0:im.shape[0], 0:im.shape[1]]
    dist = np.hypot(xx - sq["cx"], yy - sq["cy"]) * cancan.UM_PER_BIN / 1000
    sm[dist > 4] = -np.inf
    iy, ix = np.unravel_index(np.argmax(sm), sm.shape)
    rows.append({"subject": subject, "offset_mm": dist[iy, ix],
                 "peak_over_4mm_ring": sm[iy, ix] / np.mean(im[(dist > 3.8) & (dist < 4.2)])})
peaks = pd.DataFrame(rows).set_index("subject").join(fits[["cell_line", "protocol", "rate", "bursts"]])
print("Offset of the YP peak from the fitted array center (mm), and peak over the mean at 4 mm:")
print(peaks.groupby(["cell_line", "protocol", "rate"])[["offset_mm", "peak_over_4mm_ring"]].describe().round(2)
      [[("offset_mm", "count"), ("offset_mm", "50%"), ("offset_mm", "max"), ("peak_over_4mm_ring", "50%")]].to_string())

In the 0.2 MHz CANCAN wells the YP peak lies 0.31 mm (CT26) and 0.36 mm (CHO) from the fitted center
(medians; at most 1.47 mm), and is 3.6 to 4.1 times the mean at 4 mm. In control wells the brightest
point of the search area lies 3.2 to 3.9 mm out (medians), at its edge. This is consistent with the
footprint fits marking the array center the paper measures from, and the CANCAN peaks being centered. The images are single time points, so there is no clock to check.

### Profiles

For every well used: the background (3 ROIs in each footprint, median over the four footprints), and
the ROI means along the two diagonals and the two lines through the center parallel to the sides, at
ROI spacings of 1 and 0.5 mm out to 7 mm (the readings of the claims file), background subtracted and
the two lines averaged at each signed distance (Methods 2.5).

In [ ]:
bg_rows, prof_rows = [], []
for subject in used.index:
    im, sq = images[subject], squares[subject]
    bg = cancan.background(im, sq)
    bg_rows.append({"subject": subject, "background": bg["background"],
                    **{f"footprint_{i}": v for i, v in enumerate(bg["per_footprint"])}})
    for spacing in (1.0, 0.5):
        for kind in ("diagonal", "sides"):
            p = cancan.profile(im, sq, cancan.distances(spacing), kind)
            for x, v in cancan.pooled_profile(p, bg["background"]).items():
                prof_rows.append({"subject": subject, "spacing": spacing, "kind": kind, "x": x, "yp": v})
bgs = pd.DataFrame(bg_rows).set_index("subject")
profiles = pd.DataFrame(prof_rows).merge(wells[["cell_line", "protocol", "rate", "bursts", "date"]],
                                         left_on="subject", right_index=True)
fp = bgs[[c for c in bgs if c.startswith("footprint")]]
spread = (fp.max(axis=1) - fp.min(axis=1)) / bgs.background
print("Background (a.u.): range over wells", round(bgs.background.min(), 1), "to", round(bgs.background.max(), 1))
print("Footprints outside the image:", int(fp.isna().sum().sum()))
print("Spread between footprints, (max - min) / background: median", round(spread.median(), 3), "max", round(spread.max(), 3),
      "in", spread.idxmax())
print("Profile ROIs outside the imaged area:", int(profiles.yp.isna().sum()))
print("Background-subtracted YP over all profile ROIs: min", round(profiles.yp.min(), 1), "max", round(profiles.yp.max(), 1))

In [ ]:
FIT_RANGES = (4, 5, 6)
meas = []
for (subject, spacing), d in profiles[profiles.kind == "diagonal"].groupby(["subject", "spacing"]):
    s = d.set_index("x").yp
    row = {"subject": subject, "spacing": spacing, "center": s.loc[0.0],
           "yp6": s[s.index.map(abs) == 6].mean(), "yp7": s[s.index.map(abs) == 7].mean()}
    for R in FIT_RANGES:
        f = cancan.fit_lorentz(s.index.values, s.values, R)
        row[f"fwhm{R}"], row[f"R{R}"], row[f"bound{R}"] = f["b"], f["R"], f["b"] >= 99.9
    meas.append(row)
meas = pd.DataFrame(meas).merge(wells[["cell_line", "protocol", "rate", "bursts", "date"]], left_on="subject", right_index=True)
meas["ratio6"], meas["ratio7"] = meas.center / meas.yp6, meas.center / meas.yp7
m1 = meas[meas.spacing == 1.0]
print("Per-well measures, ROI spacing 1 mm (YP in a.u. after background; FWHM in mm by fit range):")
print(m1.drop(columns=["spacing"]).round(2).to_string(index=False))
print("\nLorentzian fits whose FWHM ran to the 100 mm bound (no peak), by group:")
print(meas.groupby(["cell_line", "protocol", "rate", "spacing"])[[f"bound{R}" for R in FIT_RANGES]].sum().to_string())

In [ ]:
grp = ["cell_line", "protocol", "rate", "bursts"]
fig, axs = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (cell, prot, rate) in zip(axs, [("CT26", "cancan", "0.2MHz"), ("CHO", "cancan", "0.2MHz"), ("CT26", "any", "1Hz")]):
    if prot == "any":
        sel = profiles[(profiles.cell_line == cell) & (profiles.rate == "1Hz")]
        keys = ["protocol"]
    else:
        sel = profiles[(profiles.cell_line == cell) & (profiles.protocol == prot) & (profiles.rate == rate)]
        keys = ["bursts"]
    sel = sel[(sel.kind == "diagonal") & (sel.spacing == 0.5)]
    for k, d in sel.groupby(keys):
        g = d.groupby("x").yp.agg(["mean", "sem"])
        ax.errorbar(g.index, g["mean"], g["sem"], label=f"{k[0]} (n={d.subject.nunique()})", capsize=2)
    ax.set_title(f"{cell} {rate} diagonal profiles, mean +/- s.e.")
    ax.set_xlabel("distance from center (mm)")
    ax.set_ylabel("YP, background subtracted (a.u.)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print("Plotted range of mean YP (a.u.):", round(profiles.groupby(["subject", "kind", "spacing", "x"]).yp.mean().min(), 1),
      "to", round(profiles.groupby(["subject", "kind", "spacing", "x"]).yp.mean().max(), 1), "; all points inside the axes")

## fig7be-center-linear-with-bursts

> In both CT26 and CHO cells, YP uptake at the array center increases linearly with the number of
> bursts (slopes different from zero at p < 0.001), except for saturation at 6 bursts (x4) in CT26 cells.

Center YP is the ROI at 0 mm. Linear fit on burst number, the 6-burst point excluded for CT26; readings
(a) individual experiments and (b) group means.

In [ ]:
fig7 = m1[(m1.protocol == "cancan") & (m1.rate == "0.2MHz")]
def center_fits(cell, bursts):
    d = fig7[(fig7.cell_line == cell) & fig7.bursts.isin(bursts)]
    g = d.groupby("bursts").center.mean()
    return {"a_individual": cancan.slope_test(d.bursts, d.center), "b_group_means": cancan.slope_test(g.index, g.values)}
print("Center YP by burst number (a.u.), mean, s.e., n, and per experiment date:")
print(fig7.groupby(["cell_line", "bursts"]).center.agg(["mean", "sem", "count"]).round(1).to_string())
print(fig7.pivot_table(index=["cell_line", "date"], columns="bursts", values="center", aggfunc="mean").round(0).to_string())
lin = {}
for cell, b in [("CT26", [1, 2, 4]), ("CHO", [1, 2, 4, 6]), ("CT26 with 6 (variant)", [1, 2, 4, 6])]:
    for reading, r in center_fits(cell.split()[0], b).items():
        lin[(cell, reading)] = r
        print(f"{cell:24s} bursts {b} {reading:14s} slope {r['slope']:8.1f} a.u./burst  s.e. {r['se']:6.1f}  "
              f"r {r['r']:.3f}  n {r['n']:2d}  p {r['p']:.2g}")

**Result: partially reproduced.** Center YP rises with burst number in both cell lines. The CHO slope
over all four burst numbers is positive with p < 0.001 under reading (a), individual experiments
(p = 2.2e-10). The CT26 slope over 1, 2 and 4 bursts (x4) is positive but its p is 0.0013 under reading
(a) and 0.024 under reading (b), so p < 0.001 holds in CHO only. CT26 has 11 wells here against the
paper's 3 to 5 per group, from two experiment dates.

## fig7b-ct26-saturation

> In CT26 cells the central YP uptake saturates at 6 bursts (x4), falling below the linear increase seen
> from 1 to 4 bursts, while CHO cells show no saturation.

The 6-burst mean against the 95% confidence band of the linear fit through 1, 2 and 4 bursts,
extrapolated to 6; readings (a) and (b) as above.

In [ ]:
sat = []
for cell in ("CT26", "CHO"):
    d = fig7[(fig7.cell_line == cell) & fig7.bursts.isin([1, 2, 4])]
    g = d.groupby("bursts").center.mean()
    obs6 = fig7[(fig7.cell_line == cell) & (fig7.bursts == 6)].center
    for reading, (x, y) in {"a_individual": (d.bursts, d.center), "b_group_means": (g.index, g.values)}.items():
        band = cancan.fit_band(x, y, [6]).iloc[0]
        sat.append({"cell_line": cell, "reading": reading, "fit_at_6": band.fit, "band_lo": band.lo, "band_hi": band.hi,
                    "observed_6_mean": obs6.mean(), "observed_6_sem": obs6.sem(), "n6": len(obs6),
                    "below_band": obs6.mean() < band.lo, "below_line": obs6.mean() < band.fit,
                    "within_band": band.lo <= obs6.mean() <= band.hi})
print(pd.DataFrame(sat).round(1).to_string(index=False))

**Result: partially reproduced.** The CT26 6-burst mean (4827 a.u.) lies below the line fitted through
1, 2 and 4 bursts (x4), extrapolated to 6 (6273), but inside its 95% band (lower edge 4234 under reading
(a), 4338 under reading (b)). The CHO 6-burst mean lies inside the band of the CHO fit under both readings,
as the claim expects.

## fig7cf-fwhm-with-bursts

> The FWHM of the central YP peak increases in proportion to the number of bursts in CT26 cells (slope
> p < 0.001) but not in CHO cells (p = 0.44).

Per-experiment Lorentzian FWHM; CT26 over 1, 2, 4 and 6 bursts (x4), CHO over 2, 4 and 6. Readings: fit
range (4, 5, 6 mm), ROI spacing (1, 0.5 mm), fit to experiments or to group means.

In [ ]:
fig7all = meas[(meas.protocol == "cancan") & (meas.rate == "0.2MHz")]
print("FWHM (mm) by burst number, mean, s.e., n (spacing 1 mm, fit range 5 mm):")
print(fig7all[fig7all.spacing == 1.0].groupby(["cell_line", "bursts"]).fwhm5.agg(["mean", "sem", "count"]).round(2).to_string())
rows = []
for spacing in (1.0, 0.5):
    for R in FIT_RANGES:
        for reading in ("individual", "group_means"):
            row = {"spacing": spacing, "fit_range": R, "reading": reading}
            for cell, b in (("CT26", [1, 2, 4, 6]), ("CHO", [2, 4, 6])):
                d = fig7all[(fig7all.cell_line == cell) & (fig7all.spacing == spacing) & fig7all.bursts.isin(b)]
                if reading == "individual":
                    r = cancan.slope_test(d.bursts, d[f"fwhm{R}"])
                else:
                    g = d.groupby("bursts")[f"fwhm{R}"].mean()
                    r = cancan.slope_test(g.index, g.values)
                row[f"{cell}_slope"], row[f"{cell}_p"] = r["slope"], r["p"]
            row["CT26_ok"] = row["CT26_slope"] > 0 and row["CT26_p"] < 0.001
            row["CHO_ok"] = row["CHO_p"] >= 0.05
            rows.append(row)
fw = pd.DataFrame(rows)
print(fw.round(4).to_string(index=False))
print("\nReadings where both parts hold:", int((fw.CT26_ok & fw.CHO_ok).sum()), "of", len(fw),
      "; CT26 part holds in", int(fw.CT26_ok.sum()), "; CHO part holds in", int(fw.CHO_ok.sum()))

**Result: reproduced.** Under the readings with 0.5 mm ROI spacing, a fit range of 5 or 6 mm and fits to
individual experiments, the CT26 slope is positive with p < 0.001 and the CHO slope has p >= 0.05
(0.064 and 0.090), as the criterion asks under one reading. The outcome depends on the reading: both
parts hold in 2 of the 12 readings. The CT26 part fails in every group-mean reading (p 0.008 to 0.029),
and the CHO part fails in four of the six individual-experiment readings (p 0.025 to 0.047), since the CHO
FWHM is larger at 6 bursts (2.65 mm) than at 2 and 4 (2.09 and 2.08 mm, fit range 5 mm, 1 mm spacing).

## fig7-sharpest-peak-one-burst

> One burst (x4) in CT26 cells gives the sharpest peak of the study, FWHM 1.1 +/- 0.1 mm, 7.3% of the
> inter-electrode gap and 2.3% of the electrode array area.

Mean per-experiment FWHM of CT26 at 1 burst (x4); gap 15.3 mm; area readings (a) disk of diameter FWHM
over the 117 mm2 square of 15.3 mm diagonals, (b) disk of radius FWHM over the 164 mm2 square of the
electrode centers, (c) disk of diameter FWHM over 164 mm2. Fit range and ROI spacing under each reading.

In [ ]:
one = fig7all[(fig7all.cell_line == "CT26") & (fig7all.bursts == 1)]
print("Per-experiment FWHM, CT26, 1 burst (x4) (mm):")
print(one.pivot_table(index="subject", columns="spacing", values=[f"fwhm{R}" for R in FIT_RANGES]).round(3).to_string())
sq_diag = 15.3 ** 2 / 2
sq_centers = cancan.SIDE_MM ** 2
rows = []
for spacing in (1.0, 0.5):
    for R in FIT_RANGES:
        v = one[one.spacing == spacing][f"fwhm{R}"]
        f = v.mean()
        area = {"a": np.pi * (f / 2) ** 2 / sq_diag * 100, "b": np.pi * f ** 2 / sq_centers * 100,
                "c": np.pi * (f / 2) ** 2 / sq_centers * 100}
        rows.append({"spacing": spacing, "fit_range": R, "fwhm_mean": f, "fwhm_sem": v.sem(), "n": len(v),
                     "gap_pct": f / 15.3 * 100, **{f"area_{k}_pct": a for k, a in area.items()},
                     "fwhm_ok": abs(f - 1.1) <= 0.1 + 1e-12, "gap_ok": abs(f / 15.3 * 100 - 7.3) <= 0.73 + 1e-12,
                     "area_ok_any": any(abs(a - 2.3) <= 0.23 for a in area.values())})
sharp = pd.DataFrame(rows)
print(sharp.round(3).to_string(index=False))
print("\nSmallest group-mean FWHM among all 0.2 MHz groups (spacing 1 mm, fit range 5 mm):")
print(fig7all[fig7all.spacing == 1.0].groupby(["cell_line", "bursts"]).fwhm5.mean().sort_values().head(3).round(3).to_string())

**Result: reproduced, under the assumption of 0.5 mm ROI spacing.** CT26 with 1 burst (x4) gives the
smallest group-mean FWHM of the dataset. With 0.5 mm spacing the FWHM is 1.09 to 1.10 mm, 7.1 to 7.2% of
the gap, and the area under reading (b) is 2.29 to 2.33%, all within tolerance for every fit range. With
1 mm spacing the FWHM is 1.00 to 1.01 mm, still within tolerance, but the gap percentage (6.55 to 6.61%)
is at the lower edge of 6.57% (below it for the 4 mm fit range) and no area reading reaches 2.07%, which would make the claim
partially reproduced. The 0.5 mm spacing is adopted as the reading the paper better supports (ROIs
"connecting electrodes' footprints", Methods 2.5, and a peak 1 mm wide that a 1 mm grid samples with
three points). The three experiments give 0.92 to 1.31 mm (0.5 mm spacing).

## fig7-offtarget-40-50-fold

> With 1 burst (x4) in CT26 cells, YP uptake near the electrodes is 40 to 50 times smaller than at the
> remote target in the center.

Ratio of center YP to YP near electrodes, in 36 to 55. Readings: near electrodes at (a) 6 or (b) 7 mm;
ratio (i) of group means or (ii) mean of per-experiment ratios; group (A) 1 burst (x4) or (B) each of
1, 2 and 4 bursts (x4).

In [ ]:
def ratio_table(d):
    out = {}
    for near in ("6", "7"):
        out[f"{near}mm_i_group_means"] = d.center.mean() / d[f"yp{near}"].mean()
        out[f"{near}mm_ii_mean_of_ratios"] = d[f"ratio{near}"].mean()
    return out
ct = m1[(m1.cell_line == "CT26") & (m1.protocol == "cancan") & (m1.rate == "0.2MHz")]
rt = pd.DataFrame({b: ratio_table(ct[ct.bursts == b]) for b in (1, 2, 4, 6)}).T
rt.index.name = "bursts"
print("CT26, 0.2 MHz: center YP over YP near electrodes")
print(rt.round(1).to_string())
print("\nPer-experiment ratios, 1 burst (x4):")
print(ct[ct.bursts == 1][["subject", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(1).to_string(index=False))
inr = (rt >= 36) & (rt <= 55)
print("\nReading A (1 burst) in 36 to 55:", inr.loc[1].to_dict())
print("Reading B (1, 2 and 4 bursts all in 36 to 55):", inr.loc[[1, 2, 4]].all().to_dict())

**Result: not reproduced.** With 1 burst (x4) the CT26 ratio at 6 mm is 31.9 (group means) and 34.8
(mean of per-experiment ratios), and at 7 mm 14.9 and 14.4, all below 36. The three experiments give 16.3,
37.4 and 50.7 at 6 mm. Under reading B the 4-burst ratios are 7.7 and 2.9, far from the range.

## fig7-cho-replicates-ct26

> CANCAN remote focusing of electroporation found in CT26 cells also holds in CHO cells, which show a
> central peak of YP uptake that grows with burst number.

For each burst number from 2 bursts (x4) up, the CHO mean center YP must exceed the mean YP at 6 and at
7 mm; together with the CHO part of fig7be-center-linear-with-bursts.

In [ ]:
cho = m1[(m1.cell_line == "CHO") & (m1.protocol == "cancan") & (m1.rate == "0.2MHz")]
g = cho.groupby("bursts")[["center", "yp6", "yp7"]].mean()
g["n"] = cho.groupby("bursts").size()
g["center_over_6"], g["center_over_7"] = g.center / g.yp6, g.center / g.yp7
g["holds"] = (g.center > g.yp6) & (g.center > g.yp7)
print(g.round(2).to_string())
r = lin[("CHO", "a_individual")]
print(f"\nCHO linear fit of center YP on burst number (individual experiments): slope {r['slope']:.1f}, p {r['p']:.2g}")

**Result: reproduced.** For 2, 4 and 6 bursts (x4) the CHO center YP exceeds YP at 6 and 7 mm by the
ratios printed above (1 burst (x4) is reported and also holds), and the CHO center YP rises linearly with
burst number with p < 0.001.

## fig4b-40-fold-at-02mhz

> At a packet rate of 0.2 MHz, YP uptake in the center was 40-fold higher than near the electrodes,
> despite a roughly 4-fold stronger electric field near the electrodes.

CT26, 2 bursts (x4), 0.2 MHz; ratio within 36 to 44 under one of the readings (a) 6 or (b) 7 mm, (i) group
means or (ii) mean of per-experiment ratios.

In [ ]:
two = ct[ct.bursts == 2]
print(two[["subject", "date", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(1).to_string(index=False))
r2 = pd.Series(ratio_table(two))
print("\n", r2.round(2).to_string())
print("\nWithin 36 to 44:", ((r2 >= 36) & (r2 <= 44)).to_dict())

**Result: not reproduced.** The ratio at 6 mm is 45.1 of group means and 52.1 as the mean of
per-experiment ratios, just above the 36 to 44 tolerance; at 7 mm it is 10.7 and 12.5. The per-experiment
ratios at 6 mm span 17.9 to 81.9. The dataset has five usable wells here where the paper has n = 3, and
the paper does not say which it used. The field strengths are not in the dataset.

## fig3a-cancan-center-enhanced

> All CANCAN protocols enhance electroporation at the center of the array, while the unipolar control
> produces electroporation that decreases toward the center.

CT26, 2 bursts (x4): group-mean ratio of center YP to YP at 6 mm and at 7 mm, above 1 for each CANCAN
rate and below 1 for the control. The dandiset has CANCAN at 1 Hz and 0.2 MHz and controls at 1 Hz only;
the 1 kHz and 10 kHz rates are missing.

In [ ]:
c2 = m1[(m1.cell_line == "CT26") & (m1.bursts == 2)]
g = c2.groupby(["protocol", "rate"])[["center", "yp6", "yp7"]].agg(["mean", "sem"])
g.columns = ["_".join(c) for c in g.columns]
g["n"] = c2.groupby(["protocol", "rate"]).size()
g["center_over_6"] = g.center_mean / g.yp6_mean
g["center_over_7"] = g.center_mean / g.yp7_mean
print(g.round(2).to_string())
print("\nPer control well:")
print(c2[c2.protocol == "control"][["subject", "center", "yp6", "yp7", "ratio6", "ratio7"]].round(2).to_string(index=False))
prof_c = profiles[(profiles.cell_line == "CT26") & (profiles.bursts == 2) & (profiles.protocol == "control")
                  & (profiles.kind == "diagonal") & (profiles.spacing == 1.0)]
gc = prof_c.assign(ax=prof_c.x.abs()).groupby("ax").yp.mean()
print("\nControl, mean YP by absolute distance (a.u.):", gc.round(0).to_dict())
print("Spearman correlation of control YP with distance 0 to 7 mm:", stats.spearmanr(gc.index, gc.values))

**Result: partially reproduced.** Both CANCAN rates present give center-to-edge ratios above 1 (1 Hz:
3.43 at 6 mm and 1.67 at 7 mm; 0.2 MHz: 45.1 and 10.7), but the control ratios are 1.10 and 1.11, where the
criterion needs them below 1. The control YP is nearly flat across the diagonal (2275 a.u. at the center,
1837 to 2085 from 1 to 7 mm; Spearman rho -0.36, p = 0.39), so the decrease toward the center is not
seen. Two of the four CANCAN rates (1 kHz, 10 kHz) are not in the dandiset.

## fig3bc-lorentzian-profile

> The decline of YP uptake sideways from the center is bell-shaped and follows a Lorentzian, with a
> nearly perfect fit within 4 mm of the center (R > 0.9, p < 0.001).

CT26, 2 bursts (x4), CANCAN rates present: Lorentzian fitted to the group-mean profile from -4 to 4 mm,
along the diagonals and along the lines parallel to the sides; R of fitted and observed values in that
range. Shown at ROI spacings of 1 mm (9 points) and 0.5 mm (17 points).

In [ ]:
def group_profile(rate, kind, spacing):
    d = profiles[(profiles.cell_line == "CT26") & (profiles.bursts == 2) & (profiles.protocol == "cancan")
                 & (profiles.rate == rate) & (profiles.kind == kind) & (profiles.spacing == spacing)]
    return d.groupby("x").yp.mean(), d.subject.nunique()
rows, gfits = [], {}
for spacing in (1.0, 0.5):
    for rate in ("1Hz", "0.2MHz"):
        for kind in ("diagonal", "sides"):
            gp, n = group_profile(rate, kind, spacing)
            f = cancan.fit_lorentz(gp.index.values, gp.values, 4)
            gfits[(spacing, rate, kind)] = (gp, f)
            rows.append({"spacing": spacing, "rate": rate, "direction": kind, "n_wells": n, "points": f["n"],
                         "a": f["a"], "fwhm": f["b"], "R": f["R"], "p": f["p"], "holds": f["R"] > 0.9 and f["p"] < 0.001})
lor = pd.DataFrame(rows)
print(lor.round(4).to_string(index=False))

**Result: reproduced.** For both rates present and both directions, the Lorentzian fitted from -4 to 4 mm
gives R of 0.93 to 0.99 with p < 0.001 at both ROI spacings (table above). The 1 kHz and 10 kHz rates are
not in the dandiset.

## fig3bc-offtarget-deviation-falls-with-rate

> Increasing the packet rate from 1 Hz to 0.2 MHz suppresses electroporation near the electrodes and at
> the perimeter between electrodes, so that the excess of YP uptake over the Lorentzian fit at larger
> distances from the center diminishes with packet rate.

Mean of observed minus fitted YP over the ROIs beyond 4 mm, in % of the fitted peak a, for the fits of
fig3bc-lorentzian-profile; smaller at 0.2 MHz than at 1 Hz in both directions.

In [ ]:
rows = []
for (spacing, rate, kind), (gp, f) in gfits.items():
    out = gp[gp.index.map(abs) > 4]
    dev = (out.values - cancan.lorentz(out.index.values, f["a"], f["b"])).mean() / f["a"] * 100
    rows.append({"spacing": spacing, "direction": kind, "rate": rate, "points_beyond_4mm": len(out), "deviation_pct_of_a": dev})
dv = pd.DataFrame(rows).pivot_table(index=["spacing", "direction"], columns="rate", values="deviation_pct_of_a")
dv["smaller_at_0.2MHz"] = dv["0.2MHz"] < dv["1Hz"]
print(dv.round(2).to_string())

**Result: reproduced.** Beyond 4 mm the YP excess over the Lorentzian fit is 1.7 to 2.7% of the fitted
peak at 0.2 MHz and 24 to 30% at 1 Hz, along the diagonals and along the lines parallel to the sides, at
both ROI spacings. Only these two rates are in the dandiset.

## fig4b-center-no-rate-effect

> YP uptake in the center of the array shows no significant effect of the packet rate.

A claim of absence. CT26, 2 bursts (x4), center YP at each rate against 1 Hz by two-sided Student's
t-tests with Bonferroni correction. Only 0.2 MHz can be compared with 1 Hz here; the correction is shown
for the one comparison available and for the paper's three (1 kHz, 10 kHz and 0.2 MHz against 1 Hz).

In [ ]:
c_hz = c2[(c2.protocol == "cancan") & (c2.rate == "1Hz")].center
c_mhz = c2[(c2.protocol == "cancan") & (c2.rate == "0.2MHz")].center
t = stats.ttest_ind(c_mhz, c_hz)
w = stats.ttest_ind(c_mhz, c_hz, equal_var=False)
print(f"1 Hz: mean {c_hz.mean():.0f} +/- {c_hz.sem():.0f} (n {len(c_hz)}); 0.2 MHz: mean {c_mhz.mean():.0f} +/- {c_mhz.sem():.0f} (n {len(c_mhz)})")
print(f"difference {c_mhz.mean() - c_hz.mean():.0f} a.u. ({(c_mhz.mean() / c_hz.mean() - 1) * 100:.0f}%); "
      f"Student t = {t.statistic:.2f}, p = {t.pvalue:.3f}; Bonferroni x1 p = {min(1, t.pvalue):.3f}, x3 p = {min(1, 3 * t.pvalue):.3f}")
print(f"Welch variant: p = {w.pvalue:.3f}")

**Result: reproduced.** The center YP at 0.2 MHz is 36% lower than at 1 Hz (1985 against 3088 a.u.),
and the t-test does not find the difference (p = 0.257 uncorrected, 0.770 with the paper's threefold
correction). With two 1 Hz wells the test has little power, and the 1 kHz and 10 kHz comparisons cannot
be made.

## fig4b-near-electrode-rate-decline

> YP uptake at 6 and 7 mm from the center (near the electrodes) declines steeply with increasing packet
> rate, so that higher rates suppress off-target electroporation despite a stronger electric field there.

CT26, 2 bursts (x4): power function of packet rate fitted to YP at 6 and at 7 mm, to group means and to
individual experiments (two rates only: 1 Hz and 0.2 MHz), and 0.2 MHz against 1 Hz by a two-sided t-test.

In [ ]:
cc = c2[c2.protocol == "cancan"].assign(rate_hz=lambda d: d.rate.map({"1Hz": 1.0, "0.2MHz": 208333.0}))
for near in ("yp6", "yp7"):
    g = cc.groupby("rate_hz")[near].mean()
    k_group = np.log(g.iloc[1] / g.iloc[0]) / np.log(g.index[1] / g.index[0])
    (c_, k_ind), _ = optimize.curve_fit(lambda x, c, k: c * x ** k, cc.rate_hz, cc[near], p0=[g.iloc[0], -0.2], maxfev=20000)
    t = stats.ttest_ind(cc[cc.rate_hz > 1][near], cc[cc.rate_hz == 1][near])
    print(f"{near}: 1 Hz {g.iloc[0]:.0f}, 0.2 MHz {g.iloc[1]:.0f} a.u. (ratio {g.iloc[1] / g.iloc[0]:.3f}); exponent, group means {k_group:.3f}, "
          f"individual experiments {k_ind:.3f}; t = {t.statistic:.2f}, p = {t.pvalue:.2g}")

**Result: reproduced.** YP at 0.2 MHz is 4.9% of YP at 1 Hz at 6 mm and 10.0% at 7 mm; the
power-function exponents are -0.246 and -0.188 under both fits, and the t-tests give p = 3.2e-06 and
0.00036. Only two rates are in the dandiset, so the power function passes through two group means.

## fig4c-fwhm-decreases-with-rate

> Higher packet rates focus electroporation into a smaller spot, FWHM decreasing from about 4 mm at 1 Hz
> to about 2 mm at 1 kHz, 10 kHz and 0.2 MHz, less than 15% of the gap between electrodes.

CT26, 2 bursts (x4); per-experiment FWHM fitted from -4 to 4 mm; ROI spacings 1 and 0.5 mm. Only 1 Hz and
0.2 MHz are in the dandiset.

In [ ]:
rows = []
for spacing in (1.0, 0.5):
    d = meas[(meas.cell_line == "CT26") & (meas.bursts == 2) & (meas.protocol == "cancan") & (meas.spacing == spacing)]
    hz, mhz = d[d.rate == "1Hz"].fwhm4, d[d.rate == "0.2MHz"].fwhm4
    t = stats.ttest_ind(mhz, hz)
    rows.append({"spacing": spacing, "fwhm_1Hz": hz.mean(), "sem_1Hz": hz.sem(), "n_1Hz": len(hz),
                 "fwhm_0.2MHz": mhz.mean(), "sem_0.2MHz": mhz.sem(), "n_0.2MHz": len(mhz),
                 "p": t.pvalue, "p_bonferroni_x3": min(1, 3 * t.pvalue),
                 "1Hz_in_3.6_4.4": abs(hz.mean() - 4) <= 0.4, "0.2MHz_in_1.8_2.2": abs(mhz.mean() - 2) <= 0.2,
                 "min_pct_of_gap": min(hz.mean(), mhz.mean()) / 15.3 * 100})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nPer-experiment FWHM at 1 Hz (spacing 1 mm, fit ranges 4, 5, 6 mm):")
print(m1[(m1.cell_line == "CT26") & (m1.rate == "1Hz") & (m1.protocol == "cancan")][["subject", "fwhm4", "fwhm5", "fwhm6"]].round(2).to_string(index=False))

**Result: partially reproduced.** The FWHM falls from 4.56 mm at 1 Hz to 2.00 mm at 0.2 MHz (1 mm
spacing; 4.54 and 1.88 mm at 0.5 mm spacing). The 0.2 MHz mean is within 1.8 to 2.2 mm and is 12 to 13%
of the gap, below 15%, but the 1 Hz mean is above the 3.6 to 4.4 mm tolerance (two wells, 3.91 and
5.21 mm). The 1 kHz and 10 kHz means and their tests, which the criterion needs, cannot be computed. The
0.2 MHz against 1 Hz difference has p = 0.009 with the threefold correction, where the paper reports 0.09.

## fig5c-irreversible-center (blocked; YP part shown)

The claim rests on Fig. 5 wells with propidium and a post-exposure Hoechst counterstain, which the
dandiset does not hold; its Hoechst images were taken before exposure. The YP part of the criterion is
shown on the CT26 0.2 MHz 2 bursts (x4) wells for reference only: center YP against YP at every distance
from 2 to 7 mm along the diagonal (ROI spacing 1 mm).

In [ ]:
pr = profiles[(profiles.cell_line == "CT26") & (profiles.bursts == 2) & (profiles.rate == "0.2MHz")
              & (profiles.protocol == "cancan") & (profiles.kind == "diagonal") & (profiles.spacing == 1.0)]
gm = pr.assign(ax=pr.x.abs()).groupby("ax").yp.mean()
print("Mean YP by absolute distance (a.u.):", gm.round(0).to_dict())
print("Center above every distance from 2 to 7 mm:", bool((gm.loc[0] > gm.loc[2:7]).all()))

## Summary

| Claim | Outcome |
|---|---|
| fig7be-center-linear-with-bursts | partially reproduced |
| fig7b-ct26-saturation | partially reproduced |
| fig7cf-fwhm-with-bursts | reproduced |
| fig7-sharpest-peak-one-burst | reproduced (assumption: 0.5 mm ROI spacing) |
| fig7-offtarget-40-50-fold | not reproduced |
| fig7-cho-replicates-ct26 | reproduced |
| fig4b-40-fold-at-02mhz | not reproduced |
| fig3a-cancan-center-enhanced | partially reproduced |
| fig3bc-lorentzian-profile | reproduced |
| fig3bc-offtarget-deviation-falls-with-rate | reproduced |
| fig4b-center-no-rate-effect | reproduced |
| fig4b-near-electrode-rate-decline | reproduced |
| fig4c-fwhm-decreases-with-rate | partially reproduced |
| fig4c-log-dependence | blocked (other: not in this dandiset, a rate series is in DANDI:001047) |
| fig5-raised-electrodes-weaker-center | blocked (other: in DANDI:001284) |
| fig5b-pr-gaussian-yp-lorentzian | blocked (other: in DANDI:001284) |
| fig5-yp-spot-slightly-larger-than-pr | blocked (other: in DANDI:001284) |
| fig5c-irreversible-center | blocked (other: in DANDI:001284) |
| fig6b-900ns-peak-not-significant | blocked (other: in DANDI:001277) |
| fig6c-normalized-decline-similar | blocked (other: in DANDI:001277) |
| fig6d-fwhm-sharper-longer-pulses | blocked (other: in DANDI:001277) |
| fig2-heating-below-44c | blocked (data missing) |
| fig1e-field-simulation | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** ROIs are placed by a program, not by hand in MetaMorph:
the array center is the center of a square of the paper's size fitted to the dark electrode footprints,
on images reduced to 16.25 um bins, with a pixel size of 1.625 um that the files do not state. Three
wells whose footprints could not be located were left out. The background is the median over the four
footprints of 3 ROIs each, where the paper uses 3 ROIs in one footprint. The ROI spacing and the fit
range are not given in the paper and are run under each reading; where they change an outcome
(fig7-sharpest-peak-one-burst) the 0.5 mm spacing is adopted as an assumption. The wells are those deposited, which
for most groups are more than the paper's n, and the paper does not say which it used.

**What was found about the dataset.** The dandiset holds the Fig. 7 burst series and 1 Hz CANCAN and
control wells, but none of the other series of the paper, which three other dandisets citing the paper
appear to hold (DANDI:001047, 001277 and 001284); 001276 does not point to them. Pixel size, electrode positions and the
experimental conditions are not in structured metadata (conditions only in free text);
`session_start_time` is the conversion date; two Hoechst files carry an undocumented second plane and one
well has a duplicated Hoechst fragment, another has no Hoechst file. The well counts exceed the paper's.